# Baseline: Single-Stage RoBERTa (No Pipeline)
Direct 19-class classification without gating, coarse routing, or Z3.
Compare against our 4-stage pipeline.

In [ ]:
!pip install -q transformers torch scikit-learn tqdm

In [ ]:
import json

import torch
from google.colab import files
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

uploaded = files.upload()

with open("merged_fallacies.json") as f:
    data = json.load(f)

labels_list = sorted(set(d["fallacy"] for d in data))
label2id = {l: i for i, l in enumerate(labels_list)}
id2label = {i: l for l, i in label2id.items()}

texts = [d["text"] for d in data]
labels = [label2id[d["fallacy"]] for d in data]

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Classes: {len(labels_list)}")

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "label": torch.tensor(self.labels[idx], dtype=torch.long)}

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
train_ds = SimpleDataset(X_train, y_train, tokenizer)
test_ds = SimpleDataset(X_test, y_test, tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=len(labels_list), id2label=id2label, label2id=label2id).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in pbar:
        out = model(batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device), labels=batch["label"].to(device))
        out.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += out.loss.item()
        pbar.set_postfix({"loss": f"{out.loss.item():.3f}"})
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
preds, truths = [], []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        truths.extend(batch["label"].numpy())

macro_f1 = f1_score(truths, preds, average="macro")
print(f"\n{'='*50}")
print(f"SINGLE-STAGE BASELINE: Macro F1 = {macro_f1:.4f}")
print("OUR 4-STAGE PIPELINE:  Macro F1 = 0.7264")
print(f"Difference: {0.7264 - macro_f1:+.4f}")
print(f"{'='*50}")
print(classification_report(truths, preds, target_names=labels_list, zero_division=0))